# Oetztaler Glacier Bayesian Forecast (lightweight HDF5)

This notebook uses a tiny monthly dataset and performs Bayesian forecasting to 2050 with calibrated seasonality and conservative long-term trend assumptions.

## Data, Variables, and Workflow

- Satellite data: **Sentinel-2 L2A** (`SENTINEL2_L2A`) from Copernicus openEO.
- Input bands used during preprocessing: `B03`, `B04`, `B08`, `B11`, and `SCL` (scene classification).
- Clear-pixel SCL classes: `4, 5, 6, 11` (vegetation, bare soil, water, snow/ice).
- Derived indices:
  - `NDVI = (B08 - B04) / (B08 + B04)`
  - `NDWI = (B03 - B08) / (B03 + B08)`
  - `NDSI = (B03 - B11) / (B03 + B11)`
- Derived fractions (monthly AOI averages):
  - `SnowFrac = mean(NDSI > 0.4)`
  - `WaterFrac = mean(NDWI > 0.1)`
- Lightweight file used here: `data/oetztal/oetztal_monthly_timeseries.h5`.
- Forecast horizon: monthly up to **2050-12**.

Model choices:
- `SnowFrac`: Bayesian trend + seasonal harmonics (oscillatory fit with stronger seasonal amplitude capture).
- `WaterFrac`: Bayesian **log-linear** model (`log(y)` linear in time), which implies an exponential trend in original space, with conservative slope prior to avoid unrealistic 50-year explosion.
- Fractions are constrained to `[0, 1]` in outputs.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

%matplotlib widget

In [ ]:
def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / '.git').exists():
            return cand
    return p

REPO_ROOT = find_repo_root(Path.cwd())
ROOT = REPO_ROOT / 'python_analysis' if (REPO_ROOT / 'python_analysis').exists() else REPO_ROOT
H5_FILE = ROOT / 'data' / 'oetztal' / 'oetztal_monthly_timeseries.h5'
FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not H5_FILE.exists():
    raise RuntimeError(
        f'Missing {H5_FILE}. Run from repo root: '
        f'"/home/vsilv/.local/share/pipx/venvs/openeo/bin/python" python_analysis/src/examples/oetztal_precompute_timeseries_hdf5.py'
    )

ds = xr.open_dataset(H5_FILE, engine='netcdf4')
vals = ds['values'].values
idx = pd.to_datetime(ds['time'].values)
cols = [str(v) for v in ds['metric'].values]
overview_df = pd.DataFrame(vals, index=idx, columns=cols).sort_index()
ds.close()

print('Loaded:', H5_FILE.relative_to(ROOT))
print('Rows/cols:', overview_df.shape)
print('File size KiB:', H5_FILE.stat().st_size / 1024.0)
display(overview_df.head())

In [ ]:
def frac_year(index: pd.DatetimeIndex) -> np.ndarray:
    return index.year + (index.month - 0.5) / 12.0

def design_trend_harmonics(index: pd.DatetimeIndex, t0: float, n_harmonics: int = 3) -> np.ndarray:
    t = frac_year(index) - t0
    cols = [np.ones_like(t), t]
    for k in range(1, n_harmonics + 1):
        cols.append(np.sin(2 * np.pi * k * t))
        cols.append(np.cos(2 * np.pi * k * t))
    return np.column_stack(cols)

def design_logexp(index: pd.DatetimeIndex, t0: float) -> np.ndarray:
    t = frac_year(index) - t0
    return np.column_stack([np.ones_like(t), t])

def bayesian_linear_posterior_draws(
    y: np.ndarray,
    X: np.ndarray,
    X_future: np.ndarray,
    prior_var: np.ndarray,
    prior_mean: np.ndarray | None = None,
    n_samples: int = 5000,
    alpha0: float = 2.0,
    beta0: float = 0.05,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    p = X.shape[1]
    if prior_var.shape != (p,):
        raise ValueError(f'prior_var shape {prior_var.shape} does not match p={p}')

    v0_inv = np.diag(1.0 / prior_var.astype(float))
    m0 = np.zeros(p, dtype=float) if prior_mean is None else prior_mean.astype(float)
    if m0.shape != (p,):
        raise ValueError(f'prior_mean shape {m0.shape} does not match p={p}')
    vn = np.linalg.inv(v0_inv + X.T @ X)
    mn = vn @ (X.T @ y + v0_inv @ m0)

    an = alpha0 + 0.5 * len(y)
    bn = beta0 + 0.5 * (y @ y + m0.T @ v0_inv @ m0 - mn.T @ np.linalg.inv(vn) @ mn)

    rng = np.random.default_rng(seed)
    lam = rng.gamma(shape=an, scale=1.0 / bn, size=n_samples)
    sigma2 = 1.0 / lam
    L = np.linalg.cholesky(vn)

    fit_mean_draws = np.empty((n_samples, X.shape[0]), dtype=np.float32)
    future_mean_draws = np.empty((n_samples, X_future.shape[0]), dtype=np.float32)
    for i in range(n_samples):
        w = mn + np.sqrt(sigma2[i]) * (L @ rng.standard_normal(p))
        fit_mean_draws[i, :] = X @ w
        future_mean_draws[i, :] = X_future @ w

    return fit_mean_draws, future_mean_draws

def summarize_draws(draws: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    mean = draws.mean(axis=0)
    p05 = np.quantile(draws, 0.05, axis=0)
    p95 = np.quantile(draws, 0.95, axis=0)
    return mean, p05, p95

In [ ]:
FORECAST_END = pd.Timestamp('2050-12-01')
N_SAMPLES = 6000

snow_obs = overview_df['SnowFrac'].dropna().astype(float)
water_obs = overview_df['WaterFrac'].dropna().astype(float)
future_idx = pd.date_range(snow_obs.index.max() + pd.offsets.MonthBegin(1), FORECAST_END, freq='MS')
t0 = float(np.mean(frac_year(snow_obs.index)))

# Snow: stronger harmonic representation + mild trend prior (less drastic over decades)
X_snow = design_trend_harmonics(snow_obs.index, t0=t0, n_harmonics=3)
Xf_snow = design_trend_harmonics(future_idx, t0=t0, n_harmonics=3)
# prior vars: [intercept, trend, sin/cos...]
snow_prior_var = np.array([1.0, 0.003, 0.20, 0.20, 0.10, 0.10, 0.05, 0.05], dtype=float)
snow_fit_draws, snow_fore_draws = bayesian_linear_posterior_draws(
    y=snow_obs.values, X=X_snow, X_future=Xf_snow, prior_var=snow_prior_var, n_samples=N_SAMPLES, seed=7
)
snow_fit_mean, _, _ = summarize_draws(snow_fit_draws)
snow_fore_mean, snow_fore_p05, snow_fore_p95 = summarize_draws(snow_fore_draws)
snow_fit_mean = np.clip(snow_fit_mean, 0.0, 1.0)
snow_fore_mean = np.clip(snow_fore_mean, 0.0, 1.0)
snow_fore_p05 = np.clip(snow_fore_p05, 0.0, 1.0)
snow_fore_p95 = np.clip(snow_fore_p95, 0.0, 1.0)

# Water: explicit exponential mean model in log-space, calibrated to ~0.5 by 2050
eps = 1e-4
X_water = design_logexp(water_obs.index, t0=t0)
Xf_water = design_logexp(future_idx, t0=t0)
water_target_2050 = 0.5
last_water_level = float(np.clip(water_obs.iloc[-1], eps, 1.0))
delta_years = float((FORECAST_END.year + (FORECAST_END.month - 0.5) / 12.0) - (water_obs.index[-1].year + (water_obs.index[-1].month - 0.5) / 12.0))
target_slope = np.log(np.clip(water_target_2050, eps, 1.0) / last_water_level) / max(delta_years, 1e-6)
target_intercept = np.log(last_water_level) - target_slope * (frac_year(pd.DatetimeIndex([water_obs.index[-1]]))[0] - t0)
water_prior_mean = np.array([target_intercept, target_slope], dtype=float)
water_prior_var = np.array([1e-4, 1e-8], dtype=float)
logy = np.log(np.clip(water_obs.values, eps, None))
water_fit_log_draws, water_fore_log_draws = bayesian_linear_posterior_draws(
    y=logy,
    X=X_water,
    X_future=Xf_water,
    prior_var=water_prior_var,
    prior_mean=water_prior_mean,
    n_samples=N_SAMPLES,
    seed=11,
)
water_fit_mean = np.clip(np.exp(water_fit_log_draws).mean(axis=0), 0.0, 1.0)
water_fore_mean = np.clip(np.exp(water_fore_log_draws).mean(axis=0), 0.0, 1.0)
water_fore_p05 = np.clip(np.quantile(np.exp(water_fore_log_draws), 0.05, axis=0), 0.0, 1.0)
water_fore_p95 = np.clip(np.quantile(np.exp(water_fore_log_draws), 0.95, axis=0), 0.0, 1.0)

print('Forecast months:', len(future_idx), '| end:', future_idx.max().date())
print('Water target @2050:', water_target_2050, '| calibrated slope:', round(target_slope, 4), 'per year')
print('Water forecast mean @2050:', round(float(water_fore_mean[-1]), 3))

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(13, 8), sharex=True, facecolor='white')

snow_interp = snow_obs.resample('MS').mean().interpolate('time')
ax[0].scatter(snow_obs.index, snow_obs.values, s=18, alpha=0.75, color='#1f77b4', label='Observed SnowFrac')
ax[0].plot(snow_interp.index, snow_interp.values, color='#1f77b4', linewidth=1.0, alpha=0.65, label='Observed interpolation')
ax[0].plot(snow_obs.index, snow_fit_mean, color='#0d3b66', linestyle='--', linewidth=1.6, label='Bayesian mean fit')
ax[0].plot(future_idx, snow_fore_mean, color='#e76f51', linewidth=1.8, label='Forecast mean')
ax[0].fill_between(future_idx, snow_fore_p05, snow_fore_p95, color='#e76f51', alpha=0.22, label='90% interval')
ax[0].set_title('Snow Fraction: Oscillatory Bayesian Model (Calibrated Amplitude)')
ax[0].set_ylabel('fraction')
ax[0].set_ylim(0.0, 1.0)
ax[0].legend(loc='upper right')

water_interp = water_obs.resample('MS').mean().interpolate('time')
ax[1].scatter(water_obs.index, water_obs.values, s=18, alpha=0.75, color='#2a9d8f', label='Observed WaterFrac')
ax[1].plot(water_interp.index, water_interp.values, color='#2a9d8f', linewidth=1.0, alpha=0.65, label='Observed interpolation')
ax[1].plot(water_obs.index, water_fit_mean, color='#1d3557', linestyle='--', linewidth=1.6, label='Bayesian mean fit (exp model)')
ax[1].plot(future_idx, water_fore_mean, color='#f4a261', linewidth=1.9, label='Forecast mean (more exponential)')
ax[1].fill_between(future_idx, water_fore_p05, water_fore_p95, color='#f4a261', alpha=0.22, label='90% interval')
ax[1].set_title('Water Fraction: Bayesian Exponential Forecast (Target ~0.5 by 2050)')
ax[1].set_ylabel('fraction')
ax[1].set_xlabel('time')
ax[1].set_ylim(0.0, 1.0)
ax[1].legend(loc='upper left')

for a in ax:
    a.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = FIG_DIR / 'oetztal_s2_bayesian_forecast_snow_water_to2050.png'
fig.savefig(fig_path, dpi=170, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', fig_path.relative_to(ROOT))

In [ ]:
other_metrics = ['NDVI', 'NDWI', 'NDSI']
other_results: dict[str, dict[str, np.ndarray]] = {}
other_interp: dict[str, pd.Series] = {}

for i, metric in enumerate(other_metrics):
    s = overview_df[metric].dropna().astype(float)
    Xm = design_trend_harmonics(s.index, t0=t0, n_harmonics=2)
    Xfm = design_trend_harmonics(future_idx, t0=t0, n_harmonics=2)
    prior_var = np.array([2.0, 0.01, 0.4, 0.4, 0.2, 0.2], dtype=float)
    fit_draws, fore_draws = bayesian_linear_posterior_draws(
        y=s.values, X=Xm, X_future=Xfm, prior_var=prior_var, n_samples=N_SAMPLES, seed=100 + i
    )
    fit_mean, _, _ = summarize_draws(fit_draws)
    fore_mean, fore_p05, fore_p95 = summarize_draws(fore_draws)
    other_results[metric] = {'fit_mean': fit_mean, 'fore_mean': fore_mean, 'p05': fore_p05, 'p95': fore_p95, 'obs': s.values}
    other_interp[metric] = s.resample('MS').mean().interpolate('time')

fig, ax = plt.subplots(3, 1, figsize=(13, 10), sharex=True, facecolor='white')
obs_colors = {'NDVI': '#264653', 'NDWI': '#457b9d', 'NDSI': '#6a994e'}
pred_colors = {'NDVI': '#e76f51', 'NDWI': '#f4a261', 'NDSI': '#bc6c25'}

for k, metric in enumerate(other_metrics):
    s = overview_df[metric].dropna().astype(float)
    r = other_results[metric]
    ax[k].scatter(s.index, s.values, s=15, alpha=0.75, color=obs_colors[metric], label=f'Observed {metric}')
    ax[k].plot(other_interp[metric].index, other_interp[metric].values, color=obs_colors[metric], linewidth=1.0, alpha=0.65, label='Observed interpolation')
    ax[k].plot(s.index, r['fit_mean'], linestyle='--', linewidth=1.5, color='#1d3557', label='Bayesian mean fit')
    ax[k].plot(future_idx, r['fore_mean'], linewidth=1.8, color=pred_colors[metric], label='Forecast mean')
    ax[k].fill_between(future_idx, r['p05'], r['p95'], color=pred_colors[metric], alpha=0.22, label='90% interval')
    ax[k].set_title(f'{metric}: Bayesian Forecast to 2050')
    ax[k].grid(True, alpha=0.3)
    ax[k].legend(loc='best')

ax[-1].set_xlabel('time')
plt.tight_layout()
fig_path2 = FIG_DIR / 'oetztal_s2_bayesian_forecast_other_indicators_to2050.png'
fig.savefig(fig_path2, dpi=170, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', fig_path2.relative_to(ROOT))